# From a scan to notes: OMR

Session 2, *The source and the notes*, part 4.

Optical music recognition (OMR) is a program that looks at the image of a page and proposes notes. It produces a draft, not a transcription: it can miss notes, misread pitches, merge bars, and it usually drops dynamics and words. Here we compare what two OMR programs, HOMR and OeMeR, made of page 8 of our source, Schumann's *Fröhlicher Landmann*.

We ran both programs before class, because each takes minutes. This notebook loads their results from `schumann-omr-recordings.zip`, which must be in the same folder as the notebook (`get_notebooks.py` puts it there). It also downloads the scan from IMSLP (9 MB), so that you can compare.

**How to use this notebook:** run the cells from top to bottom with Shift+Enter. Everyone does parts 1 to 3; parts 4 and 5 are extensions.

## Setup

The first cell defines helper functions; it is collapsed because you do not need to read it. The second unpacks the recorded results into the folder `schumann_omr_work` next to this notebook.

In [ ]:
# Helper functions for this notebook. Run this cell; you do not need to read it.
# They follow the same rules as the demos on the course page.

"""Portable helpers for the Schumann OMR reference notebook (standard library)."""

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET


def prepare_schumann_image(pdf, destination, *, allow_download=False):
    """Fetch/check the frozen witness and render its eighth PDF page locally.

    Poppler's pdftoppm must be installed separately. Source media never enter
    the recorded notebook bundle. PNG bytes may differ across Poppler versions.
    """
    import urllib.request
    pdf, destination = Path(pdf).resolve(), Path(destination).resolve()
    expected = "893ab03c376724a9f4474497130eb2d8510ba9d71f05bb270c8f6000b731e289"
    url = "https://s9.imslp.org/files/imglnks/usimg/0/0a/IMSLP51556-PMLP02707-RS67.pdf"
    if not pdf.exists():
        if not allow_download:
            raise FileNotFoundError("Set allow_download=True or supply the frozen IMSLP PDF.")
        pdf.parent.mkdir(parents=True, exist_ok=True)
        temporary = pdf.with_suffix(".download")
        with urllib.request.urlopen(url, timeout=60) as response, temporary.open("wb") as out:
            shutil.copyfileobj(response, out)
        if file_sha256(temporary) != expected:
            temporary.unlink()
            raise ValueError("The downloaded PDF differs from the frozen witness.")
        temporary.replace(pdf)
    if file_sha256(pdf) != expected:
        raise ValueError("Wrong witness PDF: SHA-256 differs.")
    destination.mkdir(parents=True, exist_ok=False)
    command = ["pdftoppm", "-f", "8", "-l", "8", "-r", "300", "-png", "-singlefile",
               str(pdf), str(destination / "schumann-page")]
    result = subprocess.run(command, capture_output=True, text=True, check=True, timeout=120)
    from PIL import Image
    page = destination / "schumann-page.png"
    with Image.open(page) as image:
        if image.size != (4167, 5617):
            raise ValueError(f"Recheck crop coordinates for changed image dimensions: {image.size}")
        image.crop((100, 500, 4000, 2280)).save(destination / "schumann-two-systems.png")
    version = subprocess.run(["pdftoppm", "-v"], capture_output=True, text=True, check=True)
    save_json(destination / "preparation.json", {
        "pdf_sha256": expected, "source_url": url, "original_pdf_page": 8, "printed_page": 8,
        "dpi": 300, "pixels": [4167, 5617], "command": command,
        "poppler_version": version.stdout + version.stderr, "render_messages": result.stderr,
        "page_sha256": file_sha256(page), "crop_pixels_left_top_right_bottom": [100, 500, 4000, 2280],
        "crop_sha256": file_sha256(destination / "schumann-two-systems.png"), "resampling": "None",
    })
    return page, destination / "schumann-two-systems.png"


def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def run_omr(engine, engine_python, image, destination, timeout=1200):
    """Run an installed recogniser in isolation and preserve its raw export/log.

    destination must be new. The recogniser can write diagnostics beside its
    input, so the image and any generated images are kept under local/.
    A process failure is recorded as a result, not silently repaired.
    """
    if engine not in {"homr", "oemer"}:
        raise ValueError("Supported engine names: homr, oemer")
    # Resolving a venv's Python symlink would select the base interpreter and
    # discard the isolated environment's packages.
    engine_python, image = Path(engine_python).absolute(), Path(image).resolve()
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=False)
    work = destination / "local"
    work.mkdir()
    input_image = work / "schumann-page.png"
    shutil.copyfile(image, input_image)
    module = "homr.main" if engine == "homr" else "oemer.ete"
    command = [str(engine_python), "-c", f"from {module} import main; main()", str(input_image)]
    env = dict(os.environ, OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2",
               CUDA_VISIBLE_DEVICES="", MPLCONFIGDIR=str(work / "mplconfig"),
               XDG_CACHE_HOME=str(work / "cache"))
    environment = subprocess.run(
        [str(engine_python), "-c", "import importlib.metadata as m,json,sys; "
         "print(json.dumps({'python':sys.version,'packages':dict(sorted((d.metadata['Name'],d.version) "
         "for d in m.distributions() if d.metadata.get('Name')))}))"],
        text=True, capture_output=True, check=True, env=env,
    )
    environment_info = json.loads(environment.stdout)
    if engine not in {name.lower() for name in environment_info["packages"]}:
        raise RuntimeError(f"{engine} is not installed in {engine_python}")
    metadata = {"engine": engine, "started_utc": datetime.now(timezone.utc).isoformat(),
                "input_sha256": file_sha256(input_image), "command": command,
                "hardware": "CPU; CUDA_VISIBLE_DEVICES is empty",
                "environment": environment_info,
                "status": "running", "corrections": "None: raw recogniser output is preserved."}
    save_json(destination / "run.json", metadata)
    start = time.monotonic()
    with (destination / "run.log").open("w", encoding="utf-8") as log:
        try:
            result = subprocess.run(command, cwd=work, env=env, stdout=log,
                                    stderr=subprocess.STDOUT, timeout=timeout)
            metadata.update(returncode=result.returncode,
                            status="completed" if result.returncode == 0 else "failed")
        except subprocess.TimeoutExpired:
            metadata.update(returncode=None, status="timeout")
    metadata["elapsed_seconds_including_first_run_downloads"] = round(time.monotonic() - start, 3)
    exports = sorted(work.glob("*.musicxml")) + sorted(work.glob("*.xml"))
    metadata["exports_found"] = [p.name for p in exports]
    if len(exports) == 1:
        output = destination / "raw.musicxml"
        shutil.copyfile(exports[0], output)
        metadata["output"] = {"file": output.name, "sha256": file_sha256(output), "bytes": output.stat().st_size}
    package = subprocess.run(
        [str(engine_python), "-c", f"import importlib.util; print(importlib.util.find_spec('{engine}').submodule_search_locations[0])"],
        text=True, capture_output=True, check=True, env=env,
    )
    package_root = Path(package.stdout.strip())
    weights = sorted(p for p in package_root.rglob("*") if p.is_file() and p.suffix in {".onnx", ".h5", ".safetensors"})
    metadata["downloaded_model_files"] = [
        {"file": str(p.relative_to(package_root)), "sha256": file_sha256(p), "bytes": p.stat().st_size}
        for p in weights
    ]
    save_json(destination / "run.json", metadata)
    return metadata


def convert_omr_to_mei(raw_xml, destination, python=sys.executable):
    """Convert a copy with CAMAT; a native importer failure cannot kill Jupyter."""
    raw_xml, destination = Path(raw_xml).resolve(), Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=False)
    command = [str(python), "-m", "camat.conversion", "--source", str(raw_xml),
               "--output-dir", str(destination), "--mei-version", "5.1"]
    with (destination / "conversion.log").open("w", encoding="utf-8") as log:
        result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, timeout=180)
    mei_files = sorted(destination.glob("*.mei"))
    record = {"command": command, "returncode": result.returncode,
              "status": "completed" if result.returncode == 0 and mei_files else "failed",
              "input_sha256": file_sha256(raw_xml),
              "outputs": [{"file": p.name, "sha256": file_sha256(p)} for p in mei_files]}
    save_json(destination / "conversion-run.json", record)
    return record


def musicxml_summary(path):
    """Structural inspection, not an accuracy metric; measures counted per part."""
    root = ET.parse(path).getroot()
    if root.tag != "score-partwise":
        raise ValueError("This inspector expects uncompressed partwise MusicXML")
    parts = []
    for part in root.findall("part"):
        measures = part.findall("measure")
        parts.append({"part": part.get("id"), "measures": len(measures),
                      "labels": [m.get("number") for m in measures],
                      "pitched_note_elements": len(part.findall(".//note/pitch")),
                      "rest_elements": len(part.findall(".//note/rest")),
                      "staff_values": sorted({e.text for e in part.findall(".//note/staff") if e.text}),
                      "voice_values": sorted({e.text for e in part.findall(".//note/voice") if e.text})})
    return {"file": Path(path).name, "parts": parts,
            "dynamics": len(root.findall(".//dynamics")),
            "slur_endpoints": len(root.findall(".//slur")),
            "written_words": [e.text for e in root.findall(".//direction-type/words") if e.text]}


def chord_after_rest_locations(path):
    """Locate pitched chord members attached to a preceding rest, for inspection.

    This is one specific structural warning, not a complete MusicXML validator.
    """
    root, found = ET.parse(path).getroot(), []
    for part in root.findall("part"):
        for measure in part.findall("measure"):
            leader = None
            for i, item in enumerate(measure):
                if item.tag != "note":
                    if item.tag in {"backup", "forward"}:
                        leader = None
                    continue
                if item.find("chord") is None:
                    leader = item
                elif leader is not None and leader.find("rest") is not None:
                    found.append({"part": part.get("id"), "measure_label": measure.get("number"),
                                  "child_index": i, "staff": item.findtext("staff", "1"),
                                  "voice": item.findtext("voice", "1")})
    return found


def musicxml_bar_notes(path, part_index=0, measure_index=0):
    """List a raw measure's events with onsets; indices are positions, not labels."""
    from fractions import Fraction
    root = ET.parse(path).getroot()
    part = root.findall("part")[part_index]
    division = 1
    for measure in part.findall("measure")[:measure_index + 1]:
        if measure.findtext("attributes/divisions"):
            division = int(measure.findtext("attributes/divisions"))
    measure = part.findall("measure")[measure_index]
    position, last_onset, events = Fraction(0), Fraction(0), []
    for item in measure:
        if item.tag in {"backup", "forward"}:
            delta = Fraction(int(item.findtext("duration", "0")), division)
            position += delta if item.tag == "forward" else -delta
        elif item.tag == "note":
            duration = Fraction(int(item.findtext("duration", "0")), division)
            is_chord = item.find("chord") is not None
            onset = last_onset if is_chord else position
            pitch = item.find("pitch")
            spelling = "rest" if pitch is None else (
                pitch.findtext("step", "?") + {"-2": "bb", "-1": "b", "0": "", "1": "#", "2": "##"}.get(pitch.findtext("alter", "0"), "?")
                + pitch.findtext("octave", "?"))
            events.append({"onset_quarters": str(onset), "duration_quarters": str(duration),
                           "pitch": spelling, "staff": item.findtext("staff", "1"),
                           "voice": item.findtext("voice", "1"), "chord": is_chord})
            if not is_chord:
                last_onset = onset
                position += duration
    return {"label": measure.get("number"), "events": events}


def render_score_to_svg(score, destination, python=sys.executable):
    """Run Verovio in a separate process and keep its exact warnings."""
    score, destination = Path(score).resolve(), Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    code = """from pathlib import Path
import sys,verovio,json
t=verovio.toolkit()
t.setOptions({'pageWidth':2100,'pageHeight':2970,'scale':40,'adjustPageHeight':True,'breaks':'auto','footer':'none','header':'none'})
if not t.loadFile(sys.argv[1]): raise SystemExit(2)
out=Path(sys.argv[2])
for page in range(1,t.getPageCount()+1): (out/f'page-{page}.svg').write_text(t.renderToSVG(page))
print(json.dumps({'verovio':t.getVersion(),'pages':t.getPageCount()}))
"""
    result = subprocess.run([str(python), "-c", code, str(score), str(destination)],
                            text=True, capture_output=True, timeout=120)
    (destination / "render.log").write_text(result.stdout + result.stderr, encoding="utf-8")
    return {"returncode": result.returncode, "pages": sorted(destination.glob("page-*.svg")),
            "messages": result.stdout + result.stderr}

"""Load the separate Schumann recordings ZIP; generated by the packaging tool."""

import hashlib
from pathlib import Path
import zipfile

SCHUMANN_RECORDINGS_FILE = "schumann-omr-recordings.zip"
SCHUMANN_RECORDINGS_SHA256 = "2b0c58ef7b61683eb5062ad76e08c0f28830f43b6d1c2a2ac23a2e522a10a9cb"


def find_schumann_recordings(start=None):
    """Find the ZIP beside a downloaded notebook or in the course checkout."""
    start = Path(start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        for candidate in (folder / SCHUMANN_RECORDINGS_FILE,
                          folder / "notebooks" / SCHUMANN_RECORDINGS_FILE,
                          folder / "sessions/02-building-an-edition/notebooks" / SCHUMANN_RECORDINGS_FILE):
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(
        "Copy schumann-omr-recordings.zip beside this notebook, or run get_notebooks.py "
        "to download both notebook and data. No recognition installation is needed.")


def unpack_schumann_recordings(bundle, destination):
    """Verify the data file and unpack it without replacing edited files."""
    bundle, destination = Path(bundle), Path(destination).resolve()
    if hashlib.sha256(bundle.read_bytes()).hexdigest() != SCHUMANN_RECORDINGS_SHA256:
        raise ValueError("Recording ZIP differs from this notebook. Download a matching pair.")
    with zipfile.ZipFile(bundle) as archive:
        entries = [((destination / name).resolve(), archive.read(name)) for name in archive.namelist()]
    for path, content in entries:
        if not path.is_relative_to(destination):
            raise ValueError("Unexpected archive path")
        if path.exists() and path.read_bytes() != content:
            raise FileExistsError(f"Preserving edited file: {path}. Choose a new WORK folder.")
    for path, content in entries:
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists():
            path.write_bytes(content)
    return destination

"""What the OMR notebook shows: the recorded results, the scan, one bar in words."""

from pathlib import Path
import json
import re
import shutil
import subprocess
import urllib.request

from IPython.display import display, HTML, Image

OMR_WORK = Path.cwd() / "schumann_omr_work"
# The same file and folder as in 02_pdf_preparation, so the scan is downloaded once.
SCAN_PDF = Path.cwd() / "schumann_edition_work/sources/imslp-51556-schumann-op68.pdf"
# The first two systems of page 8 (pickup and bars 1–8) at 150 dpi: half of the
# 300 dpi crop (100, 500, 4000, 2280) that HOMR read.
TWO_SYSTEMS = ["-r", "150", "-x", "50", "-y", "250", "-W", "1950", "-H", "890"]
OMR_LENGTHS = {"1/4": "sixteenth", "1/2": "eighth", "3/4": "dotted eighth", "1": "quarter",
               "3/2": "dotted quarter", "2": "half", "3": "dotted half", "4": "whole"}
OMR_STEPS = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}


def load_schumann_recordings():
    """Unpack the recordings ZIP next to the notebook and check every file."""
    folder = unpack_schumann_recordings(find_schumann_recordings(),
                                        OMR_WORK / f"recorded-{SCHUMANN_RECORDINGS_SHA256[:8]}")
    manifest = json.loads((folder / "manifest.json").read_text())
    for relative, expected in manifest["files_sha256"].items():
        if file_sha256(folder / relative) != expected:
            raise ValueError(f"{relative} in {folder} has changed. Delete that folder and run the cell again.")
    print("OMR results loaded:", len(manifest["files_sha256"]), "files in", folder)
    return folder


def get_schumann_scan(recorded):
    """Download IMSLP 51556 once and check that it is the course's copy."""
    source = json.loads((recorded / "source.json").read_text())
    if not SCAN_PDF.exists():
        print("Downloading the scan from IMSLP (9 MB) ...")
        SCAN_PDF.parent.mkdir(parents=True, exist_ok=True)
        temporary = SCAN_PDF.with_suffix(".download")
        request = urllib.request.Request(source["source_url"], headers={"User-Agent": "camat-schumann-seminar"})
        try:
            with urllib.request.urlopen(request, timeout=90) as response, temporary.open("wb") as out:
                shutil.copyfileobj(response, out)
        except OSError as problem:
            raise RuntimeError(f"Could not download the scan from IMSLP ({problem}). Check the internet "
                               f"connection and run the cell again, or put a copy of the file here: {SCAN_PDF}") from None
        if file_sha256(temporary) != source["pdf_sha256"]:
            temporary.unlink()
            raise ValueError("The download is not the scan the course uses. Run the cell again later.")
        temporary.replace(SCAN_PDF)
    elif file_sha256(SCAN_PDF) != source["pdf_sha256"]:
        raise ValueError(f"{SCAN_PDF} is not the scan the course uses. Move it away and run the cell again.")
    return SCAN_PDF


def show_schumann_scan(recorded, width=900):
    pdf = get_schumann_scan(recorded)
    image = OMR_WORK / "scan-two-systems.png"
    if not image.exists():
        if not shutil.which("pdftoppm"):
            raise RuntimeError("Poppler's pdftoppm is missing: select the seminar kernel (setup page, step 7).")
        OMR_WORK.mkdir(parents=True, exist_ok=True)
        subprocess.run(["pdftoppm", "-f", "8", "-l", "8", *TWO_SYSTEMS, "-png", "-singlefile",
                        str(pdf), str(image.with_suffix(""))], check=True, capture_output=True, timeout=120)
    display(Image(filename=str(image), width=width))


# The score on a white background (readable in dark themes), without Verovio's
# bar numbers: they count the pickup as bar 1, and the print has none.
def show_omr_score(svg_file):
    svg = Path(svg_file).read_text(encoding="utf-8")
    svg = svg[svg.index("<svg"):]
    svg = re.sub(r'<g [^>]*class="mNum[^"]*"[^>]*>.*?</g>', "", svg, flags=re.S)
    display(HTML(f'<div style="background:white; padding:4px; overflow-x:auto">{svg}</div>'))


def _omr_event(pitches, duration):
    if not pitches:
        return f"rest {OMR_LENGTHS.get(duration, duration + ' quarter notes')}"
    def height(name):
        return 12 * int(name[-1]) + OMR_STEPS[name[0]] + name.count("♯") - name.count("♭") - 2 * name.count("𝄫")
    chord = "+".join(sorted(pitches, key=height))
    return f"{chord} {OMR_LENGTHS.get(duration, duration + ' quarter notes')}"


def read_bar(recorded, bar, run="homr-two-systems"):
    """Print one bar of an OMR result in words. bar: "pickup", or 1, 2, ... as musicians count."""
    raw = recorded / run / "raw.musicxml"
    index = 0 if str(bar).lower() == "pickup" else int(bar)
    count = musicxml_summary(raw)["parts"][0]["measures"]
    if not 0 <= index < count:
        print(f"This result has the pickup and bars 1–{count - 1}.")
        return
    found = musicxml_bar_notes(raw, measure_index=index)
    print("Pickup" if index == 0 else f"Bar {index}", "as HOMR read it" if run.startswith("homr") else "")
    for staff, name in (("1", "treble"), ("2", "bass")):
        events = []
        for event in found["events"]:
            if event["staff"] != staff:
                continue
            pitch = event["pitch"].replace("bb", "𝄫").replace("b", "♭").replace("#", "♯")
            if event["chord"] and events:
                events[-1][0].append(pitch)
            else:
                events.append(([] if pitch == "rest" else [pitch], event["duration_quarters"]))
        print(f"  {name + ':':8}", ", ".join(_omr_event(p, d) for p, d in events))

In [ ]:
RECORDED = load_schumann_recordings()

## 1. The scan

The top of page 8: the first two systems, with the pickup and bars 1–8. The print has no bar numbers; we count as musicians do: the pickup, then bar 1 for the first full bar.

The cell downloads the scan once and keeps it in the folder `schumann_edition_work/sources`, where `02_pdf_preparation.ipynb` finds it later.

In [ ]:
show_schumann_scan(RECORDED)

## 2. HOMR's result for the same two systems

We gave HOMR only this part of the page, cut out of a picture of the whole page. HOMR wrote the notes it found as MusicXML; Verovio draws them below. Verovio chooses its own line breaks, so the systems do not end where the print's do.

Compare the two pictures: the notes, but also the words and signs above and below the staves.

In [ ]:
show_omr_score(RECORDED / "homr-two-systems/rendered/page-1.svg")

## 3. Bar by bar

The next cell lists one bar of HOMR's result in words, for each staff. Notes that sound together are joined with `+`. Middle C is C4.

1. Look at bar 1 and bar 5 in the bass: which pitch does HOMR read at the end of each bar? Compare it with the scan.
2. Change `BAR` to another bar (`"pickup"`, or 1 to 8) and run the cell again.
3. Find one thing HOMR got wrong or left out.

In [ ]:
BAR = 1  # "pickup", or 1 to 8
read_bar(RECORDED, BAR)

<details><summary>What we found</summary>

- **Bars 1 and 5, bass:** HOMR reads F3 at the end of both bars, like the print. The MEI file from the MEI Sample Collection, on which our starting file is based, has A3 there. That is how we found the wrong note (part 3 on the course page).
- **The notes are right:** every pitch and every note length in these two systems.
- **Words and dynamics are missing:** the tempo words *Frisch und munter.* and the *f* (forte) at the start of both systems.
- **Beams are missing:** the print joins eighth notes with beams; in HOMR's result every eighth has its own flag.
- **Slurs are missing or in the wrong place.** HOMR missed the start of the slur over bars 3 and 4. The other mistakes come from Verovio, the program that draws HOMR's file: in bar 2, for example, the file puts the slur on the bass notes, as in the print, but Verovio draws it from the treble. A mistake can come from any program on the way, not only from the OMR.

So even a good OMR result needs a comparison with the scan, note by note and sign by sign.

</details>

## 4. Extension: the whole page

### HOMR

We also gave HOMR the whole page. It found 21 bars, the right number: the pickup and 20 bars. But in bar 2 it put the F4 of the bass into the treble chord above it, so the bass of that bar adds up to only 3½ quarter notes. And Verovio crashes when it reads this file, so there is no score to show: in bar 9 and later, HOMR joined notes to rests as if they sounded together, and Verovio cannot handle that.

In [ ]:
summary = musicxml_summary(RECORDED / "homr-fullpage/raw.musicxml")
print("Bars HOMR found on the whole page:", summary["parts"][0]["measures"])
read_bar(RECORDED, 2, run="homr-fullpage")

### OeMeR

OeMeR is a second OMR program. On our computer it ran only with older versions of some of the libraries it uses; then it took about three minutes for the page. For most of the page it reads the right hand in the bass clef, so those notes come out far too low, and at first it misses the key signature (one flat). It found 19 bars instead of 21, has a rest where the print has the bass C3 of the pickup, and has no dynamics or slurs. Compare its score with the scan: a file that Verovio reads without complaint can still be full of recognition errors.

In [ ]:
summary = musicxml_summary(RECORDED / "oemer-cpu/raw.musicxml")
print("Bars OeMeR found on the whole page:", summary["parts"][0]["measures"])
show_omr_score(RECORDED / "oemer-cpu/rendered/page-1.svg")

Each run has its own folder in `schumann_omr_work/recorded-…`. Its `run.log` shows what the program printed while it worked, also for two OeMeR attempts that stopped with an error before reading the page.

## 5. Extension: run OMR yourself

Not in class: this needs a fast internet connection, several GB of disk space, the program [uv](https://docs.astral.sh/uv/) and minutes per page. The cells below repeat what we did before class. Each switch turns on one step:

- `PREPARE_NEW_INPUT` draws page 8 as an image at 300 dpi and cuts out the first two systems;
- `INSTALL_ENGINE_ENVIRONMENTS` installs HOMR and OeMeR, each in its own Python environment, with the exact versions we used;
- `RUN_FRESH_OMR` runs HOMR on the page and on the two systems, and OeMeR on the page;
- `CONVERT_FRESH_OUTPUTS` converts each MusicXML result to MEI with CAMAT.

Each new run goes into a new folder (`NEW`); nothing you already have is overwritten.

In [ ]:
PREPARE_NEW_INPUT = False
INSTALL_ENGINE_ENVIRONMENTS = False
RUN_FRESH_OMR = False
CONVERT_FRESH_OUTPUTS = False

NEW = OMR_WORK / "fresh-01"  # Choose a new name for another try.
UV = shutil.which("uv")
ENGINES = ("homr", "oemer")

In [ ]:
if PREPARE_NEW_INPUT:
    full_image, cropped_image = prepare_schumann_image(get_schumann_scan(RECORDED), NEW / "local/input")
    print(full_image, cropped_image)
else:
    print("Off. Set PREPARE_NEW_INPUT = True to draw page 8 again.")

In [ ]:
engine_pythons = {}
for engine in ENGINES:
    venv = NEW / "environments" / engine
    python = venv / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    engine_pythons[engine] = python
    recorded_env = "homr-fullpage" if engine == "homr" else "oemer-cpu"
    requirements = RECORDED / recorded_env / "requirements.txt"
    commands = [
        [UV or "uv", "venv", "--python", "3.11", str(venv)],
        [UV or "uv", "pip", "install", "--python", str(python), "--no-deps", "-r", str(requirements)],
    ]
    for command in commands:
        print(subprocess.list2cmdline(command))
    if INSTALL_ENGINE_ENVIRONMENTS:
        if not UV:
            raise RuntimeError("Install uv first.")
        if venv.exists():
            raise FileExistsError(f"Keeping the existing environment: {venv}")
        for command in commands:
            subprocess.run(command, check=True)

In [ ]:
fresh_runs = []
if RUN_FRESH_OMR:
    inputs = [("homr", "fullpage", "schumann-page.png"),
              ("homr", "two-systems", "schumann-two-systems.png"),
              ("oemer", "fullpage", "schumann-page.png")]
    for engine, scope, filename in inputs:
        if engine not in ENGINES:
            continue
        destination = NEW / "runs" / f"{engine}-{scope}"
        record = run_omr(engine, engine_pythons[engine], NEW / "local/input" / filename, destination)
        print(engine, scope, record["status"], record.get("output"))
        fresh_runs.append(destination)
else:
    print("Off. The recorded results above stay available.")

In [ ]:
CAMAT_PYTHON = sys.executable  # The Python that has CAMAT; in the seminar environment, this one.
if CONVERT_FRESH_OUTPUTS:
    for run_folder in fresh_runs:
        raw = run_folder / "raw.musicxml"
        if raw.exists():
            print(convert_omr_to_mei(raw, run_folder / "converted", python=CAMAT_PYTHON))
else:
    print("Off. Set CONVERT_FRESH_OUTPUTS = True after a fresh run.")

## Where this leads

For our edition we start from an existing MEI encoding of the whole piece, not from an OMR result. We check it against the scan just as carefully: comparing it with HOMR's reading is how we found the wrong note in bars 1 and 5.

Links: [HOMR](https://github.com/liebharc/homr), [OeMeR](https://github.com/BreezeWhite/oemer), [Verovio](https://www.verovio.org), [CAMAT](https://github.com/egorpol/camat_v2), the [IMSLP scan](https://imslp.org/wiki/Special:ImagefromIndex/51556), and the [MEI encoding](https://github.com/music-encoding/sample-encodings/blob/e4a3c4dece588ae0bdb13e5cf956340dcd99b447/MEI_5.1/Music/Complete_examples/Schumann_Landmann_Op68_No10.mei) by Maja Hartwig in the MEI Sample Collection.